<a href="https://colab.research.google.com/github/WJ714/Decoupling_ET-GPP/blob/temperature_masking/geoFMs4EC.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
%%bash
# 1) Upgrade pip & install core libraries
pip install --upgrade pip setuptools wheel
pip install torch torchvision timm einops rasterio earthengine-api huggingface_hub

# 2) Clone the HF repo with source & weights
git clone https://huggingface.co/ibm-nasa-geospatial/Prithvi-EO-2.0-300M /content/Prithvi-EO-2.0-300M

cd /content/Prithvi-EO-2.0-300M

# 3) Reshape into a Python package
mkdir -p prithvi_mae_pkg
mv prithvi_mae.py inference.py prithvi_mae_pkg/

# 4) Expose the module
cat > prithvi_mae_pkg/__init__.py << 'EOF'
from .prithvi_mae import PrithviMAE
EOF

# 5) Add PEP517 build files
cat > pyproject.toml << 'EOF'
[build-system]
requires = ["setuptools>=42","wheel"]
build-backend = "setuptools.build_meta"
EOF

cat > setup.cfg << 'EOF'
[metadata]
name = prithvi-mae
version = 2.0.300M
description = Prithvi-EO-2.0-300M Masked Autoencoder

[options]
packages = find:
install_requires =
    torch
    torchvision
    timm
    einops
    rasterio
    earthengine-api
    huggingface_hub
include_package_data = true
EOF

# 6) Install in editable mode
pip install -e .

# 7) Download the model checkpoint
python - << 'EOF'
from huggingface_hub import snapshot_download
snapshot_download("ibm-nasa-geospatial/Prithvi-EO-2.0-300M", local_dir=".")
EOF

echo "✅ Prithvi-EO-2.0-300M installed. You can now:"
echo "   from prithvi_mae import PrithviMAE"
echo "   model = PrithviMAE(**config)"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 17.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 42.1 MB/s eta 0:00:00
  Attempting uninstall: setuptools
    Found existing installation: setuptools 75.2.0
    Uninstalling setuptools-75.2.0:
      Successfully uninstalled setuptools-75.2.0
  Attempting uninstall: pip
    Found existing installation: pip 24.1.2
    Uninstalling pip-24.1.2:
      Successfully uninstalled pip-24.1.2
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 37.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 164.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 157.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 47.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 14.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 71.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ipython 7.34.0 requires jedi>=0.16, which is not installed.
Cloning into '/content/Prithvi-EO-2.0-300M'...
  error: subprocess-exited-with-error
  
  × Getting requirements to build editable did not run successfully.
  │ exit code: 1
  ╰─> See above for output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
error: subprocess-exited-with-error

× Getting requirements to build editable did not run successfully.
│ exit code: 1
╰─> See above for output.

note: This error originates from a subprocess, and is likely not a problem with pip.
Fetching 13 files: 100%|██████████| 13/13 [00:01<00:00,  9.58it/s]


In [5]:
cd Prithvi-EO-2.0-300M

/content/Prithvi-EO-2.0-300M


In [6]:
from prithvi_mae import PrithviMAE

In [7]:
from prithvi_mae import PrithviMAE
import json

# Load config
with open('/content/Prithvi-EO-2.0-300M/config.json') as f:
    cfg = json.load(f)
# Override to 300M dims:
cfg.update({
  "embed_dim": 1024, "depth": 24, "num_heads": 16,
  "decoder_embed_dim": 512, "decoder_depth": 8, "decoder_num_heads": 16
})

model = PrithviMAE(**cfg)

PrithviMAE(
  (encoder): PrithviViT(
    (patch_embed): PatchEmbed(
      (proj): Conv3d(6, 1024, kernel_size=(1, 16, 16), stride=(1, 16, 16))
      (norm): Identity()
    )
    (blocks): ModuleList(
      (0-23): 24 x Block(
        (norm1): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
        (attn): Attention(
          (qkv): Linear(in_features=1024, out_features=3072, bias=True)
          (q_norm): Identity()
          (k_norm): Identity()
          (attn_drop): Dropout(p=0.0, inplace=False)
          (proj): Linear(in_features=1024, out_features=1024, bias=True)
          (proj_drop): Dropout(p=0.0, inplace=False)
        )
        (ls1): Identity()
        (drop_path1): Identity()
        (norm2): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
        (mlp): Mlp(
          (fc1): Linear(in_features=1024, out_features=4096, bias=True)
          (act): GELU(approximate='none')
          (drop1): Dropout(p=0.0, inplace=False)
          (norm): Identity()
         

In [8]:
import torch
ckpt = torch.load('/content/Prithvi-EO-2.0-300M/Prithvi_EO_V2_300M.pt', map_location='cpu')
state = ckpt.get('model', ckpt.get('state_dict', ckpt))
model.load_state_dict(state)
model.eval()
print("✅ Model ready!")

✅ Model ready!


In [9]:
from google.colab import auth
auth.authenticate_user()    # this will open a link; paste the code into the cell prompt

import ee
ee.Authenticate()
ee.Initialize(project='horizontal-cab-458915-t5')

In [10]:
import datetime, math
import numpy as np
import torch
import pandas as pd
import io

In [11]:
sites_csv = """\
id,lat,lon
CA-Cbo,44.3184,-79.9341
CA-DBB,49.1293,-122.9849
CA-ER1,43.6405,-80.4123
US-Akn,33.3825,-81.5653
US-ALQ,46.0308,-89.6067
US-ARM,36.6058,-97.4888
US-Bar,44.0646,-71.2881
US-Bi1,38.0992,-121.4993
US-Bi2,38.1091,-121.5351
US-BRG,39.2167,-86.5406
US-CF1,46.7815,-117.0821
US-CF2,46.7840,-117.0908
US-CF3,46.7551,-117.1261
US-CF4,46.7518,-117.1285
US-GLE,41.3665,-106.2399
US-Ha1,42.5378,-72.1715
US-Ho1,45.2041,-68.7402
US-Ho2,45.2091,-68.7470
US-Jo1,32.5820,-106.6350
US-Jo2,32.5849,-106.6032
US-Me2,44.4526,-121.5589
US-Me6,44.3233,-121.6078
US-MMS,39.3232,-86.4131
US-Mo1,39.2298,-92.1167
US-Mo3,39.2322,-92.1493
US-MOz,38.7441,-92.2000
US-Mpj,34.4385,-106.2377
US-Myb,38.0499,-121.7650
US-NC3,35.7990,-76.6560
US-NC4,35.7879,-75.9038
US-Ne1,41.1651,-96.4766
US-ONA,27.3836,-81.9509
US-Pnp,43.0896,-89.4158
US-Rls,43.1439,-116.7356
US-Rms,43.0645,-116.7486
US-Ro4,44.6781,-93.0723
US-Ro5,44.6910,-93.0576
US-Ro6,44.6946,-93.0578
US-Rwf,43.1207,-116.7231
US-Rws,43.1675,-116.7132
US-Seg,34.3623,-106.7020
US-Ses,34.3349,-106.7442
US-Sne,38.0369,-121.7547
US-SP1,29.7381,-82.2188
US-SRG,31.7894,-110.8277
US-SRM,31.8214,-110.8661
US-SSH,40.6658,-77.9041
US-Syv,46.2420,-89.3477
US-Ton,38.4309,-120.9660
US-Tw1,38.1074,-121.6469
US-Tw4,38.1027,-121.6413
US-UMB,45.5598,-84.7138
US-UMd,45.5625,-84.6975
US-Var,38.4133,-120.9508
US-Vcm,35.8884,-106.5321
US-Vcp,35.8642,-106.5967
US-Whs,31.7438,-110.0522
US-Wjs,34.4255,-105.8615
US-Wkg,31.7365,-109.9419
US-xAB,45.7624,-122.3303
US-xAE,35.4106,-99.0588
US-xBR,44.0639,-71.2873
US-xCL,33.4012,-97.5700
US-xCP,40.8155,-104.7456
US-xDC,47.1617,-99.1066
US-xDL,32.5417,-87.8039
US-xDS,28.1250,-81.4362
US-xGR,35.6890,-83.5019
US-xHA,42.5369,-72.1727
US-xJE,31.1948,-84.4686
US-xJR,32.5907,-106.8425
US-xKA,39.1104,-96.6129
US-xKZ,39.1008,-96.5631
US-xMB,38.2483,-109.3883
US-xML,37.3783,-80.5248
US-xNG,46.7697,-100.9154
US-xNQ,40.1776,-112.4524
US-xRM,40.2759,-105.5459
US-xSB,29.6893,-81.9934
US-xSE,38.8901,-76.5600
US-xSL,40.4619,-103.0293
US-xSR,31.9107,-110.8355
US-xST,45.5089,-89.5864
US-xTA,32.9505,-87.3933
US-xTR,45.4937,-89.5857
US-xUK,39.0404,-95.1921
US-xUN,46.2339,-89.5373
US-xWD,47.1282,-99.2414
"""

In [12]:
# 2) Read into DataFrame
sites = pd.read_csv(io.StringIO(sites_csv))

# 3) Extraction parameters
years  = range(2017, 2021)
months = range(1, 13)
patch_px, px_m = 128, 30
half_m = patch_px * px_m / 2.0

def mask_hls(img):
    f = img.select('Fmask')
    ok = (
        f.bitwiseAnd(1<<1).eq(0)  # no clouds
       .And(f.bitwiseAnd(1<<3).eq(0))  # no shadows
       .And(f.bitwiseAnd(1<<4).eq(0))  # no snow/ice
       .And(f.rightShift(6).bitwiseAnd(3).eq(1))  # low aerosol
    )
    return img.updateMask(ok)

records = []
for _, row in sites.iterrows():
    site_id, lat, lon = row.id, row.lat, row.lon

    # Precompute the 128×128px lon/lat box
    deg_lat =  half_m / 111320.0
    deg_lon =  half_m / (111320.0 * math.cos(math.radians(lat)))
    coords = [
        [lon - deg_lon, lat - deg_lat],
        [lon + deg_lon, lat - deg_lat],
        [lon + deg_lon, lat + deg_lat],
        [lon - deg_lon, lat + deg_lat],
        [lon - deg_lon, lat - deg_lat],
    ]
    region = ee.Geometry.Polygon([coords], geodesic=False)

    for year in years:
        for m in months:
            start = f'{year:04d}-{m:02d}-01'
            end_dt = datetime.date(year, m, 1) + datetime.timedelta(days=32)
            end   = end_dt.replace(day=1).isoformat()

            # Filter + mask
            col = (
                ee.ImageCollection("NASA/HLS/HLSL30/v002")
                  .filterDate(start, end)
                  .filterBounds(region)
                  .select(['B2','B3','B4','B5','B6','B7','Fmask'])
                  .map(mask_hls)
            )
            if col.size().getInfo() == 0:
                continue  # no valid scenes

            # Median composite & fill
            img = col.median().unmask(0)
            proj30 = img.projection().atScale(px_m)
            img30  = img.reproject(proj30)
            props  = img30.sampleRectangle(region=region, defaultValue=0).getInfo()['properties']
            arr    = np.stack([np.array(props[b]) for b in ['B2','B3','B4','B5','B6','B7']],axis=0)
            # pad/crop to 128×128
            _, H, W = arr.shape
            arr = np.pad(arr, ((0,0),(0,max(0,patch_px-H)),(0,max(0,patch_px-W))), constant_values=0)
            arr = arr[:,:patch_px,:patch_px]

            # Encode & median-pool tokens
            inp = torch.from_numpy(arr).float().unsqueeze(0).unsqueeze(2)
            julian = datetime.date(year,m,15).timetuple().tm_yday
            tc = torch.tensor([[[year, julian]]], dtype=torch.float32)
            lc = torch.tensor([[lat, lon]],      dtype=torch.float32)
            with torch.no_grad():
                tokens,_,_ = model.encoder(inp, tc, lc, mask_ratio=0.0)
            patch_toks = tokens[0,1:,:]
            median_tok = patch_toks.median(dim=0).values.cpu().numpy()

            rec = {'site': site_id, 'year': year, 'month': m}
            rec.update({f'tok_{i}': float(median_tok[i]) for i in range(median_tok.shape[0])})
            records.append(rec)

# 4) Save results
df = pd.DataFrame.from_records(records)
df.to_csv('monthly_prithvi_tokens_2017_2020.csv', index=False)
print("✔️ Finished:", df.shape, "→ monthly_prithvi_tokens_2017_2020.csv")

✔️ Finished: (4135, 1027) → monthly_prithvi_tokens_2017_2020.csv


In [13]:
df

,site,year,month,tok_0,tok_1,tok_2,tok_3,tok_4,tok_5,tok_6,...,tok_1014,tok_1015,tok_1016,tok_1017,tok_1018,tok_1019,tok_1020,tok_1021,tok_1022,tok_1023
0,CA-Cbo,2017,1,-0.184805,0.225137,-0.223782,-0.051311,-0.172221,0.596590,-0.175561,...,-0.017591,-0.034253,-0.073798,0.172888,-0.043468,0.279252,-0.280675,-0.020926,-0.223292,0.024875
1,CA-Cbo,2017,2,-0.184805,0.225138,-0.223786,-0.051309,-0.172221,0.596592,-0.175560,...,-0.017592,-0.034253,-0.073796,0.172888,-0.043466,0.279252,-0.280674,-0.020927,-0.223292,0.024875
2,CA-Cbo,2017,3,-0.184804,0.225139,-0.223784,-0.051310,-0.172220,0.596596,-0.175561,...,-0.017590,-0.034253,-0.073797,0.172887,-0.043468,0.279252,-0.280674,-0.020927,-0.223294,0.024876
3,CA-Cbo,2017,4,-0.184806,0.225138,-0.223786,-0.051310,-0.172220,0.596594,-0.175560,...,-0.017592,-0.034254,-0.073796,0.172889,-0.043468,0.279251,-0.280674,-0.020927,-0.223292,0.024876
4,CA-Cbo,2017,5,-0.184805,0.225138,-0.223784,-0.051309,-0.172219,0.596593,-0.175559,...,-0.017589,-0.034253,-0.073796,0.172887,-0.043468,0.279252,-0.280674,-0.020927,-0.223293,0.024875
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4130,US-xWD,2020,8,-0.184806,0.225139,-0.223783,-0.051311,-0.172219,0.596595,-0.175562,...,-0.017590,-0.034253,-0.073797,0.172887,-0.043468,0.279252,-0.280675,-0.020930,-0.223293,0.024875
4131,US-xWD,2020,9,-0.212493,0.154605,-0.066512,-0.157088,-0.267252,0.962834,-0.092389,...,-0.101750,0.152568,-0.049824,0.044164,0.020105,0.098835,-0.059771,0.058741,-0.166558,0.108026
4132,US-xWD,2020,10,-0.219535,0.179061,-0.076380,-0.143782,-0.214891,0.721025,-0.091942,...,-0.071640,0.194387,-0.016972,-0.003649,0.049014,0.104130,0.006644,0.091093,-0.175987,0.102230
4133,US-xWD,2020,11,-0.250371,0.150510,-0.038802,-0.149676,-0.218202,0.702667,-0.079468,...,-0.068723,0.216231,0.017759,-0.033332,0.072654,0.089188,0.056297,0.049791,-0.138538,0.081345


In [14]:
import os
assert os.path.exists('monthly_prithvi_tokens_2017_2020.csv'), \
    "monthly_prithvi_tokens_2017_2020.csv not found! Run the extraction step first."

import pandas as pd
import numpy as np
from sklearn.decomposition import PCA
from joblib import Parallel, delayed

# Load
df = pd.read_csv('monthly_prithvi_tokens_2017_2020.csv')

# Feature matrix
feature_cols = [c for c in df.columns if c.startswith('tok_')]
X = df[feature_cols].values

# Fit PCA
pca = PCA(n_components=12, random_state=42)
pca.fit(X)

# Parallel transform
n_jobs = 4
indices = np.array_split(np.arange(X.shape[0]), n_jobs)
pcs_list = Parallel(n_jobs=n_jobs)(
    delayed(lambda idxs: pca.transform(X[idxs]))(idxs) for idxs in indices
)
X12 = np.vstack(pcs_list)

# Append & save
for i in range(12):
    df[f'PC{i+1}'] = X12[:, i]
df.to_csv('monthly_prithvi_tokens_pca12.csv', index=False)

print("✅ PCA completed, saved to monthly_prithvi_tokens_pca12.csv")
print("Explained variance ratios:", pca.explained_variance_ratio_)

✅ PCA completed, saved to monthly_prithvi_tokens_pca12.csv
Explained variance ratios: [0.4569925  0.1303755  0.0953868  0.06090006 0.03801387 0.02215142
 0.01709729 0.01440332 0.01340644 0.00991228 0.00915181 0.00820593]


In [15]:
df

,site,year,month,tok_0,tok_1,tok_2,tok_3,tok_4,tok_5,tok_6,...,PC3,PC4,PC5,PC6,PC7,PC8,PC9,PC10,PC11,PC12
0,CA-Cbo,2017,1,-0.184805,0.225137,-0.223782,-0.051311,-0.172221,0.596590,-0.175561,...,-0.292764,-0.433159,0.335899,-0.134801,-0.006792,-0.061219,-0.001188,0.059941,0.022118,-0.093195
1,CA-Cbo,2017,2,-0.184805,0.225138,-0.223786,-0.051309,-0.172221,0.596592,-0.175560,...,-0.292767,-0.433158,0.335901,-0.134798,-0.006792,-0.061215,-0.001189,0.059940,0.022114,-0.093195
2,CA-Cbo,2017,3,-0.184804,0.225139,-0.223784,-0.051310,-0.172220,0.596596,-0.175561,...,-0.292765,-0.433158,0.335899,-0.134799,-0.006791,-0.061219,-0.001189,0.059939,0.022116,-0.093198
3,CA-Cbo,2017,4,-0.184806,0.225138,-0.223786,-0.051310,-0.172220,0.596594,-0.175560,...,-0.292764,-0.433157,0.335900,-0.134800,-0.006793,-0.061218,-0.001187,0.059942,0.022116,-0.093198
4,CA-Cbo,2017,5,-0.184805,0.225138,-0.223784,-0.051309,-0.172219,0.596593,-0.175559,...,-0.292767,-0.433157,0.335896,-0.134800,-0.006789,-0.061215,-0.001192,0.059943,0.022116,-0.093197
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4130,US-xWD,2020,8,-0.184806,0.225139,-0.223783,-0.051311,-0.172219,0.596595,-0.175562,...,-0.292765,-0.433157,0.335900,-0.134802,-0.006796,-0.061218,-0.001187,0.059942,0.022115,-0.093197
4131,US-xWD,2020,9,-0.212493,0.154605,-0.066512,-0.157088,-0.267252,0.962834,-0.092389,...,-0.995263,-0.225118,0.629573,0.467082,0.782296,-0.244570,0.056746,-0.157998,-0.050040,0.536297
4132,US-xWD,2020,10,-0.219535,0.179061,-0.076380,-0.143782,-0.214891,0.721025,-0.091942,...,-0.585039,-0.819913,0.400113,0.660633,0.411196,-0.190268,0.060283,-0.311877,0.197728,0.589170
4133,US-xWD,2020,11,-0.250371,0.150510,-0.038802,-0.149676,-0.218202,0.702667,-0.079468,...,-0.283060,-0.842395,0.394911,1.149559,-0.176496,-0.540162,0.011287,-0.126617,0.528923,0.668057
